## 1. Setup

In [126]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [127]:
import pandas as pd
import pickle
from src.recommender import MovieRecommender


In [170]:
from src.mood_classifier import MoodClassifier
from src.genre_mapper import MOOD_GENRES
from src.movie_filter import MovieFilter

## 2. Load Data & TF-IDF

In [128]:
movies = pd.read_csv("../data/processed/imdb_preprocessed.csv")

movies.head()

,movie_name,year,genre,tags
0,jawan,2023,action thriller,action thriller highoctan action thriller outl...
1,jaane jaan,2023,crime drama mystery,crime drama mysteri singl mother daughter comm...
2,jailer,2023,action comedy crime,action comedi crime retir jailer goe manhunt f...
3,rocky aur rani kii prem kahaani,2023,comedy drama family,comedi drama famili flamboy punjabi rocki inte...
4,omg 2,2023,comedy drama,comedi drama unhappi civilian ask court mandat...


In [172]:
from src.movie_search import MovieSearch

movie_search = MovieSearch(movies)

In [129]:
with open("../models/tfidf_matrix.pkl", "rb") as file:
    tfidf_matrix = pickle.load(file)

##  3. Initialize Recommender

In [130]:
recommender = MovieRecommender(
    movies,
    tfidf_matrix
)

## 4. Basic recommendation test

In [131]:
user_mood = "I am feeling stressed and tired"

favourite_movie = "Jawan"

result = recommender.recommend(
    mood_input=user_mood,
    favourite_movie=favourite_movie,
    top_n=10
)

Predicted Mood : relaxed
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'drama']
Matched movie: jawan (80.0%)
Matched Movie: jawan


In [132]:
print("Result type:")
print(type(result))

print("\nResult:")
print(result)

Result type:
<class 'dict'>

Result:
{'predicted_mood': 'relaxed', 'matched_movie': 'jawan', 'recommendations':            movie_name  year                   genre  \
53       om shanti om  2007     action comedy drama   
154   chennai express  2013     action comedy drama   
721        the intern     V   comedy drama thriller   
1711    finding fanny  2014  adventure comedy drama   
264        gehraiyaan  2022           drama romance   
120         godfather  2022      action crime drama   
253         mumbaikar  2023   action drama thriller   
201   merry christmas  2023          drama thriller   
109     singham again  2024                   drama   
1921          satya 2  2013      action crime drama   

                                                   tags  similarity_score  
53    action comedi drama 1970 om aspir actor murder...          0.184466  
154   action comedi drama man travel rameshwaram via...          0.180890  
721   comedi drama thriller indian adapt intern 2015..

In [133]:
print("Predicted Mood:")
print(result["predicted_mood"])

Predicted Mood:
relaxed


In [134]:
print("Matched Movie:")
print(result["matched_movie"])

Matched Movie:
jawan


In [135]:
recommendations = result["recommendations"]

recommendations

,movie_name,year,genre,tags,similarity_score
53,om shanti om,2007,action comedy drama,action comedi drama 1970 om aspir actor murder...,0.184466
154,chennai express,2013,action comedy drama,action comedi drama man travel rameshwaram via...,0.180890
721,the intern,V,comedy drama thriller,comedi drama thriller indian adapt intern 2015...,0.180842
1711,finding fanny,2014,adventure comedy drama,adventur comedi drama man embark road trip fin...,0.171228
264,gehraiyaan,2022,drama romance,drama romanc take journey deep root intricaci ...,0.170956
120,godfather,2022,action crime drama,action crime drama death polit leader mysteri ...,0.165413
253,mumbaikar,2023,action drama thriller,action drama thriller stori base life peopl mu...,0.164570
201,merry christmas,2023,drama thriller,drama thriller christma eve unev day turn worl...,0.161827
109,singham again,2024,drama,drama plot wrap rohit shetti kareena kapoor de...,0.160719
1921,satya 2,2013,action crime drama,action crime drama build strong underworld man...,0.154478


## 5. Similarity Evaluation


In [136]:
recommendations[
    ["movie_name", "similarity_score"]
].sort_values("similarity_score", ascending=False)

,movie_name,similarity_score
53,om shanti om,0.184466
154,chennai express,0.180890
721,the intern,0.180842
1711,finding fanny,0.171228
264,gehraiyaan,0.170956
120,godfather,0.165413
253,mumbaikar,0.164570
201,merry christmas,0.161827
109,singham again,0.160719
1921,satya 2,0.154478


In [137]:
recommendations["similarity_score"].describe()

count    10.000000
mean      0.169539
std       0.009945
min       0.154478
25%       0.162513
50%       0.168185
75%       0.178438
max       0.184466
Name: similarity_score, dtype: float64

In [138]:
test_movies = [
    "Jawan",
    "Interstellar",
    "Titanic",
    "Toy Story",
    "The Dark Knight"
]

In [139]:
test_mood = "I am feeling happy"

evaluation_results = []

for movie in test_movies:

    try:

        result = recommender.recommend(
            mood_input=test_mood,
            favourite_movie=movie,
            top_n=10
        )

        recommendations = result["recommendations"]

        evaluation_results.append({
            "input_movie": movie,
            "matched_movie": result["matched_movie"],
            "predicted_mood": result["predicted_mood"],
            "recommendation_count": len(recommendations),
            "average_similarity": recommendations[
                "similarity_score"
            ].mean(),
            "max_similarity": recommendations[
                "similarity_score"
            ].max()
        })

    except Exception as e:

        evaluation_results.append({
            "input_movie": movie,
            "matched_movie": None,
            "predicted_mood": None,
            "recommendation_count": 0,
            "average_similarity": None,
            "max_similarity": None,
            "error": str(e)
        })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: war (72.0%)
Matched Movie: war
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']


,input_movie,matched_movie,predicted_mood,recommendation_count,average_similarity,max_similarity,error
0,Jawan,jawan,happy,2,0.021294,0.025754,NaN
1,Interstellar,war,happy,2,0.011628,0.015311,NaN
2,Titanic,None,None,0,NaN,NaN,No movie found similar to 'Titanic'.
3,Toy Story,None,None,0,NaN,NaN,No movie found similar to 'Toy Story'.
4,The Dark Knight,None,None,0,NaN,NaN,No movie found similar to 'The Dark Knight'.


In [140]:
evaluation_df[
    [
        "input_movie",
        "matched_movie",
        "average_similarity",
        "max_similarity"
    ]
]

,input_movie,matched_movie,average_similarity,max_similarity
0,Jawan,jawan,0.021294,0.025754
1,Interstellar,war,0.011628,0.015311
2,Titanic,None,NaN,NaN
3,Toy Story,None,NaN,NaN
4,The Dark Knight,None,NaN,NaN


In [141]:
overall_average_similarity = (
    evaluation_df["average_similarity"]
    .dropna()
    .mean()
)

print(
    f"Overall Average Similarity: "
    f"{overall_average_similarity:.2%}"
)

Overall Average Similarity: 1.65%


In [142]:
mood_test_cases = [
    "I am feeling very happy today",
    "I feel sad and lonely",
    "I am feeling stressed and want something relaxing",
    "I want something exciting and intense",
    "I am scared and want something frightening",
    "I want a romantic movie tonight",
    "I feel motivated and want something inspiring"
]

In [143]:
mood_test_data = [
    ("I am feeling very happy today", "happy"),
    ("I feel sad and lonely", "sad"),
    ("I am stressed and tired", "relaxed"),
    ("I want something scary", "scared"),
    ("I want a romantic movie tonight", "romantic"),
    ("I need motivation to achieve my goals", "motivated"),
    ("I want to think about life and its meaning", "thoughtful"),
    ("I want an intense action movie", "excited"),
    ("I want something completely random", "neutral")
]

In [ ]:

mood_results = []

for mood_input in mood_test_cases:

    try:

        result = recommender.recommend(
            mood_input=mood_input,
            favourite_movie="Jawan",
            top_n=10
        )

        mood_results.append({
            "input": mood_input,
            "predicted_mood": result["predicted_mood"],
            "matched_movie": result["matched_movie"],
            "recommendation_count": len(
                result["recommendations"]
            )
        })

    except Exception as e:

        mood_results.append({
            "input": mood_input,
            "predicted_mood": None,
            "matched_movie": None,
            "recommendation_count": 0,
            "error": str(e)
        })

mood_evaluation_df = pd.DataFrame(mood_results)

mood_evaluation_df

Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']


Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : sad
Mood Confidence : 100%
Selected Genres : ['drama', 'romance']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : relaxed
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'drama']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : happy
Mood Confidence : 50%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : relaxed
Selected Genres : ['comedy', 'family', 'drama']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : romantic
Mood Confidence : 100%
Selected Genres : ['romance', 'drama']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : relaxed
Selected Genres : ['comedy', 'family', 'drama']
Matched movie: jawan (80.0%)
Matched Movie: jawan


,input,predicted_mood,matched_movie,recommendation_count
0,I am feeling very happy today,happy,jawan,2
1,I feel sad and lonely,sad,jawan,10
2,I am feeling stressed and want something relaxing,relaxed,jawan,10
3,I want something exciting and intense,happy,jawan,2
4,I am scared and want something frightening,relaxed,jawan,10
5,I want a romantic movie tonight,romantic,jawan,10
6,I feel motivated and want something inspiring,relaxed,jawan,10


## 6. Mood Classification Evaluation

In [148]:
classifier = MoodClassifier()

movie_filter = MovieFilter(movies)

## 7. Candidate Generation Evaluation

In [149]:
candidate_results = []

for mood_input in mood_test_cases:

    prediction = classifier.predict(mood_input)

    if isinstance(prediction, dict):
        mood = prediction["mood"]
    else:
        mood = prediction

    genres = MOOD_GENRES.get(mood)

    if genres is None:
        candidate_results.append({
            "input": mood_input,
            "predicted_mood": mood,
            "keywords": None,
            "candidate_count": 0
        })
        continue

    
    candidate_movies = movie_filter.filter_by_genres(genres)

    candidate_results.append({
        "input": mood_input,
        "predicted_mood": mood,
        "keywords": ", ".join(genres),
        "candidate_count": len(candidate_movies)
    })

candidate_df = pd.DataFrame(candidate_results)

candidate_df

,input,predicted_mood,keywords,candidate_count
0,I am feeling very happy today,happy,"comedy, family, adventure",2
1,I feel sad and lonely,sad,"drama, romance",1552
2,I am feeling stressed and want something relaxing,relaxed,"comedy, family, drama",1552
3,I want something exciting and intense,happy,"comedy, family, adventure",2
4,I am scared and want something frightening,relaxed,"comedy, family, drama",1552
5,I want a romantic movie tonight,romantic,"romance, drama",1552
6,I feel motivated and want something inspiring,relaxed,"comedy, family, drama",1552


In [150]:
candidate_df[
    [
        "predicted_mood",
        "candidate_count"
    ]
]

,predicted_mood,candidate_count
0,happy,2
1,sad,1552
2,relaxed,1552
3,happy,2
4,relaxed,1552
5,romantic,1552
6,relaxed,1552


In [151]:
candidate_df["candidate_count"].describe()

count       7.000000
mean     1109.142857
std       756.322557
min         2.000000
25%       777.000000
50%      1552.000000
75%      1552.000000
max      1552.000000
Name: candidate_count, dtype: float64

In [155]:
result = recommender.recommend(
    mood_input="I am feeling happy",
    favourite_movie="Jawan",
    top_n=10
)

recommendations = result["recommendations"]

recommendations[
    ["movie_name", "tags"]
]

Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: jawan (80.0%)
Matched Movie: jawan


,movie_name,tags
826,102 not out,comedi drama 102 2018 comedydrama film legenda...
1428,jaggu ki lalten,drama jaggu ki lalten comedydrama music film e...


In [156]:
duplicate_count = (
    recommendations["movie_name"]
    .duplicated()
    .sum()
)

print(
    "Duplicate recommendations:",
    duplicate_count
)

Duplicate recommendations: 0


In [157]:
from collections import Counter

all_tags = []

for tags in recommendations["tags"].dropna():

    tag_list = str(tags).split()

    all_tags.extend(tag_list)

tag_counts = Counter(all_tags)

tag_counts.most_common(10)

[('kapoor', 3),
 ('drama', 2),
 ('comedydrama', 2),
 ('film', 2),
 ('amitabh', 2),
 ('bachchan', 2),
 ('play', 2),
 ('rishi', 2),
 ('comedi', 1),
 ('102', 1)]

### 8. Recommendation Diversity

In [158]:
top_similarity_results = []

for movie in test_movies:

    try:

        result = recommender.recommend(
            mood_input="I am feeling happy",
            favourite_movie=movie,
            top_n=10
        )

        recs = result["recommendations"]

        top_similarity_results.append({
            "movie": movie,
            "top_1_similarity": recs.iloc[0]["similarity_score"],
            "top_5_average": recs.head(5)["similarity_score"].mean(),
            "top_10_average": recs["similarity_score"].mean()
        })

    except Exception as e:

        print(f"Error for {movie}: {e}")

similarity_df = pd.DataFrame(top_similarity_results)

similarity_df

Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: jawan (80.0%)
Matched Movie: jawan
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: war (72.0%)
Matched Movie: war
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Error for Titanic: No movie found similar to 'Titanic'.
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Error for Toy Story: No movie found similar to 'Toy Story'.
Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Error for The Dark Knight: No movie found similar to 'The Dark Knight'.


,movie,top_1_similarity,top_5_average,top_10_average
0,Jawan,0.025754,0.021294,0.021294
1,Interstellar,0.015311,0.011628,0.011628


In [159]:
top_movie = recommendations.iloc[0]

print("Top Recommendation")
print("-------------------")

print("Movie:", top_movie["movie_name"])
print("Year:", top_movie["year"])
print(
    "Similarity:",
    f"{top_movie['similarity_score']:.2%}"
)

Top Recommendation
-------------------
Movie: 102 not out
Year: 2018
Similarity: 2.58%


## 9. Fuzzy Search Evaluation

In [161]:
movie_search.find_movie("Jawan")

Matched movie: jawan (80.0%)


'jawan'

In [162]:
movie_search.find_movie("Jwan")

Matched movie: wanted (77.1%)


'wanted'

In [163]:
invalid_movie = movie_search.find_movie(
    "xyzrandommovie12345"
)

print(invalid_movie)

None


In [164]:
test_movies = [
    "Jawan",
    "Interstellar",
    "Titanic",
    "Toy Story",
    "The Dark Knight"
]

## 10. Baseline Comparison

In [165]:
from sklearn.metrics.pairwise import cosine_similarity

def baseline_recommend(
    favourite_movie,
    top_n=10
):

    favourite_index = movie_search.find_movie_index(
        favourite_movie
    )

    if favourite_index is None:
        raise ValueError(
            f"Movie not found: {favourite_movie}"
        )

    favourite_vector = tfidf_matrix[favourite_index]

    similarity_scores = cosine_similarity(
        favourite_vector,
        tfidf_matrix
    ).flatten()

    similarity_scores[favourite_index] = -1

    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    recommendations = movies.iloc[top_indices].copy()

    recommendations["similarity_score"] = (
        similarity_scores[top_indices]
    )

    return recommendations

In [166]:
baseline = baseline_recommend(
    "Jawan",
    top_n=10
)

baseline[
    ["movie_name", "year", "similarity_score"]
]

Matched movie: jawan (80.0%)


,movie_name,year,similarity_score
191,happy new year,2014,0.215500
8,pathaan,2023,0.206956
76,don,2006,0.200807
53,om shanti om,2007,0.184466
154,chennai express,2013,0.180890
721,the intern,V,0.180842
51,fighter,2024,0.174950
1711,finding fanny,2014,0.171228
264,gehraiyaan,2022,0.170956
120,godfather,2022,0.165413


In [167]:
mood_based = recommender.recommend(
    mood_input="I am feeling happy",
    favourite_movie="Jawan",
    top_n=10
)

mood_based["recommendations"][
    ["movie_name", "year", "similarity_score"]
]

Predicted Mood : happy
Mood Confidence : 100%
Selected Genres : ['comedy', 'family', 'adventure']
Matched movie: jawan (80.0%)
Matched Movie: jawan


,movie_name,year,similarity_score
826,102 not out,2018,0.025754
1428,jaggu ki lalten,2022,0.016835


## 11. Final Summary

In [168]:
print("===================================")
print("   MOVIE RECOMMENDER EVALUATION")
print("===================================")

print(
    "\nDataset size:",
    len(movies)
)

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)

print(
    "Movies tested:",
    len(test_movies)
)

print(
    "Mood inputs tested:",
    len(mood_test_cases)
)

print(
    "Average recommendation similarity:",
    f"{overall_average_similarity:.2%}"
)

print(
    "Duplicate recommendations:",
    duplicate_count
)

print("===================================")

   MOVIE RECOMMENDER EVALUATION

Dataset size: 2199
TF-IDF matrix shape: (2199, 5000)
Movies tested: 5
Mood inputs tested: 7
Average recommendation similarity: 1.65%
Duplicate recommendations: 0


In [169]:
evaluation_df.to_csv(
    project_root / "data" / "evaluation_results.csv",
    index=False
)

mood_evaluation_df.to_csv(
    project_root / "data" / "mood_evaluation_results.csv",
    index=False
)

candidate_df.to_csv(
    project_root / "data" / "candidate_evaluation_results.csv",
    index=False
)

print("Evaluation results saved.")

Evaluation results saved.
